# District Beta Diversity (Jaccard)

Compares district-level species composition turnover after spatial thinning.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, leaves_list
sns.set_theme(style='whitegrid', context='paper')
RANDOM_SEED = 42


In [ ]:
p = next((x for x in [Path('file6.csv'), Path('../file6.csv'), Path('../../file6.csv')] if x.exists()), None)
if p is None:
    raise FileNotFoundError('file6.csv not found')
cols = ['stateProvince','verbatimScientificName','decimalLatitude','decimalLongitude','eventDate']
df = pd.read_csv(p, usecols=cols, low_memory=False).dropna(subset=cols)
for c in ['decimalLatitude','decimalLongitude']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df = df.dropna(subset=['decimalLatitude','decimalLongitude'])
df['stateProvince'] = df['stateProvince'].astype(str).str.strip()
df['verbatimScientificName'] = df['verbatimScientificName'].astype(str).str.strip()


In [ ]:
lon0 = float(df['decimalLongitude'].median())
lat0 = float(df['decimalLatitude'].median())
lat_rad = np.radians(df['decimalLatitude'].to_numpy())
df['x_km'] = (df['decimalLongitude'].to_numpy() - lon0) * 111.320 * np.cos(lat_rad)
df['y_km'] = (df['decimalLatitude'].to_numpy() - lat0) * 110.574
df['grid_x'] = np.floor(df['x_km'] / 5.0).astype(int)
df['grid_y'] = np.floor(df['y_km'] / 5.0).astype(int)
df['cell_id'] = df['stateProvince'].astype(str) + '|' + df['grid_x'].astype(str) + '_' + df['grid_y'].astype(str)
df['_key'] = df['cell_id'] + '|' + df['verbatimScientificName']
rng = np.random.default_rng(RANDOM_SEED)
df['_r'] = rng.random(len(df))
df_thin = df.sort_values('_r').groupby('_key', as_index=False).head(1)


In [ ]:
presence = pd.crosstab(df_thin['stateProvince'], df_thin['verbatimScientificName'])
presence = (presence > 0).astype(int)
dist = pdist(presence.values, metric='jaccard')
jaccard = pd.DataFrame(squareform(dist), index=presence.index, columns=presence.index)
jaccard.head()


In [ ]:
link = linkage(dist, method='average')
order = leaves_list(link)
ordered = jaccard.iloc[order, order]
plt.figure(figsize=(10,8))
sns.heatmap(ordered, cmap='mako', linewidths=0.3)
plt.title('District-to-district Jaccard dissimilarity (thinned presence)')
plt.tight_layout()
plt.show()


In [ ]:
pairs = jaccard.where(np.triu(np.ones(jaccard.shape), k=1).astype(bool)).stack().reset_index()
pairs.columns = ['district_a','district_b','jaccard_dissimilarity']
pairs.sort_values('jaccard_dissimilarity', ascending=False).head(20)
